# Part teòrica

**Variables:** Files i columnes (caselles del taulell)

**Domini:** Paraules del diccionari

**Restriccions:**
- Files i columnes han d'estar dins del taulell
- Files i columnes > 1 casella
- Interseccions de files i columnes tenen la mateixa lletra
- Si es troba una # acaba la fila o la columna
- Les paraules han d'estar escrites de dalt a baix i d'esquerra a dreta
- No es pot repetir una paraula
- Les paraules han de tenir una llargada menor o igual al màxim de m o n del taulell

**Tamany espai de solucions inicial:** k^(mxn) on m son les files, n les columnes del taulell y k les lletres de l'alfabet

**Estratègia de millora:** Fer ús d'una heurística per a decidir quina assignació fer a continuació. Per exemple triar una paraula amb el major nombre de restriccions respecte a altres no asignades, ja que si té un major impacte sobre la resta pot portar a acabar abans


# Codi

## Carregar biblioteques

In [210]:
import numpy as np


# **Exercici 1**

**Data loading**

In [211]:
def loadCrossword(file):
    data = []
    with open(file, 'r') as file:
        for line in file.readlines():
            elements = line.strip().split()
            data.append(elements)
    return(np.array(data))

def loadDictionary(file):
    words = {}
    with open(file, 'r') as file:
        for word in file.readlines():    #Dic
            
            if len(word.strip()) not in words.keys():
                words[len(word.strip())] = [word.strip()]
            else:
                v = words[len(word.strip())]
                v.append(word.strip())

    return words


In [212]:
#variable = [[[pos_inicial], len, v/h, interseccions]] vertical = 1, horitzontal = 0 #intersections

def cercaVariablesHoritzontal(taulell, variables):
    for n, i in enumerate(taulell):
        length = 0
        initial_pos = [n, 0]
        for m, j in enumerate(i):
            if j == '#':
                if length > 1:
                    variables.append([initial_pos, length, 0])
                length = 0
                initial_pos = [n, m + 1]
            elif m == len(i) - 1:
                length += 1
                if length > 1:
                    variables.append([initial_pos, length, 0])
                length = 0
                initial_pos = [n, m + 1]
            else: length += 1

**Cerca de variables**

In [213]:
def cercaVariablesVertical(taulell, variables):
    transposed_taulell = taulell.transpose()
    for n, i in enumerate(transposed_taulell):
        length = 0
        initial_pos = [n, 0]
        for m, j in enumerate(i):
            if j == '#':
                if length > 1:
                    initial_pos.reverse()
                    variables.append([initial_pos, length, 1])
                length = 0
                initial_pos = [n, m + 1]
            elif m == len(i) - 1:
                length += 1
                if length > 1:
                    initial_pos.reverse()
                    variables.append([initial_pos, length, 1])
                length = 0
                initial_pos = [n, m + 1]
            else: length += 1

In [214]:
def cercaVariables(taulell, variables):
    cercaVariablesHoritzontal(taulell, variables)
    cercaVariablesVertical(taulell, variables)

In [215]:
def calculaPosicionsVariable(variable):
    variable_positions = []
    for x in range(variable[1]):
        i, j = 0, 0
        if variable[2] == 0: i = x
        else: j = x
        variable_positions.append([variable[0][0]+j, variable[0][1]+i])
    
    return variable_positions

**Interseccions**

In [216]:
def interseccions(variables):
    for variable in variables:
        intersections = []
        for pos in calculaPosicionsVariable(variable):
            if pos not in intersections:
                intersections.append(pos)
        variable.append(intersections)

**Dist**

In [217]:
#variable = [[[pos_inicial], len, v/h, interseccions]] vertical = 1, horitzontal = 0
def intersectLetter(variable, word, assigned_variable, pos):
    if variable[2] == 0:
        return assigned_variable[1][pos[0] - assigned_variable[0][0][0]] == word[pos[1] - variable[0][1]]
    elif variable[2] == 1:
        return assigned_variable[1][pos[1] - assigned_variable[0][0][1]] == word[pos[0] - variable[0][0]]
    else: return False

**Comprovació de restriccions**

In [218]:
def isValid(word, variable, assigned_variables, taulell):
    if len(word) != variable[1]:
        return False

    for assigned_variable in assigned_variables:    #nomes implementar interseccions
        if word == assigned_variable[1]:
            return False
    '''
    aux_taulell = copy.deepcopy(taulell)

    for var, assigned_word in assigned_variables:
        positions = calculaPosicionsVariable(var)
        for i, j in positions:
            aux_taulell[i][j] = assigned_word[i - var[0][0] if var[2] == 1 else j - var[0][1]]

    for position, letter in zip(calculaPosicionsVariable(variable), word):
        i, j = position
        if aux_taulell[i][j] != '0' and aux_taulell[i][j] != letter:
            return False
    '''
    for var in assigned_variables:
        for intersection in variable[3]:
            if intersection in var[0][3] and not intersectLetter(variable, word, var, intersection): return False
    return True 

**Backtracking**

```
Funcio Backtracking(LVA,LVNA,R,D)
    Si (LVNA és buida) llavors Retornar(LVA) fSi
    Var=Cap(LVNA);
    Per a cada (valor del Domini(Var, D) que podem assignar a Var) fer
        Si (SatisfaRestriccions([Var valor],LVA,R)) llavors
            Res=Backtracking(Insertar([Var, valor],LVA),Cua(LVNA),R,D);
            Si (Res és una solució completa) llavors
                Retornar(Res);
            Fsi
        Fsi
    Fper
    Retornar(Falla)
FFuncio

```

In [219]:
def backtracking(assigned_variables, variables, taulell, diccionary):
    if not variables:
        return assigned_variables

    var = variables[0]

    for word in diccionary[var[1]]:
        if isValid(word, var, assigned_variables, taulell):
            assigned_variables.append([var, word])
            result = backtracking(assigned_variables, variables[1:], taulell, diccionary)

            if result:
                return result

            assigned_variables.pop()  

    return None

**Print taulell final**

In [220]:
def printSolution(assigned_variables, taulell):
    for variable, word in assigned_variables:
        positions = calculaPosicionsVariable(variable)
        for i, j in positions:
            taulell[i][j] = word[i - variable[0][0] if variable[2] == 1 else j - variable[0][1]]

    for row in taulell:
        print(" ".join(row))

**Main**

In [221]:
if __name__ == '__main__':
    
    taulell = loadCrossword('crossword_CB_v3.txt')
    dictionary = loadDictionary('diccionari_CB_v3.txt')
    
    assigned_variables = []
    variables = []
    cercaVariables(taulell, variables)
    
    interseccions(variables)
    
    res = backtracking(assigned_variables, variables, taulell, dictionary)

    print("Resultat: ", res)
    
    printSolution(assigned_variables, taulell)
        
    
    

Resultat:  [[[[0, 0], 6, 0, [[0, 0], [0, 1], [0, 2], [0, 3], [0, 4], [0, 5]]], 'CANTAR'], [[[2, 2], 4, 0, [[2, 2], [2, 3], [2, 4], [2, 5]]], 'CLAN'], [[[4, 1], 5, 0, [[4, 1], [4, 2], [4, 3], [4, 4], [4, 5]]], 'PREMI'], [[[5, 0], 4, 0, [[5, 0], [5, 1], [5, 2], [5, 3]]], 'PIAR'], [[[6, 0], 2, 0, [[6, 0], [6, 1]]], 'ON'], [[[0, 0], 4, 1, [[0, 0], [1, 0], [2, 0], [3, 0]]], 'CARA'], [[[5, 0], 2, 1, [[5, 0], [6, 0]]], 'PO'], [[[4, 1], 3, 1, [[4, 1], [5, 1], [6, 1]]], 'PIN'], [[[4, 2], 2, 1, [[4, 2], [5, 2]]], 'RA'], [[[0, 3], 6, 1, [[0, 3], [1, 3], [2, 3], [3, 3], [4, 3], [5, 3]]], 'TALLER'], [[[0, 5], 5, 1, [[0, 5], [1, 5], [2, 5], [3, 5], [4, 5]]], 'RANCI']]
C A N T A R
A # # A # A
R # C L A N
A # # L # C
# P R E M I
P I A R # #
O N # # # #
